# Classification: Gradient Boosting Classifier (GBC)

## Justification of Preprocessing Strategy

### Scale Invariance

The Gradient Boosting Classifier is an ensemble of Decision Trees. Since each split in these trees is based on finding a threshold for a single feature, the algorithm is invariant to the scale of the inputs. Whether the data is standardized, normalized, or in its original form, the resulting decision boundaries remain identical. Consequently, we will use the **Original Data** to preserve computational efficiency and clinical interpretability.

### Iterative Error Correction

GBC builds trees sequentially. Each new tree attempts to minimize the loss function (error) of the previous ensemble using a gradient-descent approach. This allows the model to progressively learn the complex patterns that distinguish diabetic from non-diabetic patients.

## Experiment Design

We defined three optimization levels to identify the most robust configuration:

- **Baseline**: use Scikit-Learn default parameters to establish a pure performance reference.
- **GridSearchCV**: systematic search (guided by previous experiments) testing combinations of `n_estimators`, `learning_rate`, `max_depth` and `subsample`
- **Optuna**: Bayesian optimization to explore a fine-grained range of critical hyperparameters and maximize overall metrics (with focus on recall and balanced performance).

In [1]:
import pandas as pd
import numpy as np
import time
import mlflow
import optuna
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import recall_score, accuracy_score, f1_score

# MLflow Configuration
mlflow.set_tracking_uri("sqlite:///C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/mlflow.db")
mlflow.set_experiment("Classification_GradientBoosting")

2026/05/07 12:21:20 INFO mlflow.tracking.fluent: Experiment with name 'Classification_GradientBoosting' does not exist. Creating a new experiment.


<Experiment: artifact_location=('file:c:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente '
 'de '
 'Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/notebooks/Classification/EnsembleMethods/Boosting/GradBoostClassifier/mlruns/10'), creation_time=1778152880249, experiment_id='10', last_update_time=1778152880249, lifecycle_stage='active', name='Classification_GradientBoosting', tags={}, workspace='default'>

In [2]:
df = pd.read_csv("C:\\Users\\Tiago Silva\\Uni\\OneDrive - Universidade Portucalense\\Ambiente de Trabalho\\Uni\\3ano2sem\\LAD\\Grupo5_ProjetoLAD_Parte2\\TrabalhoLAD\\data\\diabetes_dataset_new_variables.csv")

categorical_cols = [
    'gender', 'ethnicity', 'smoking_status', 'education_level',
    'employment_status', 'age_groups', 'weight_status', 'income_level'
]

# Apply One-Hot Encoding
df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

X = df_final.drop(["diagnosed_diabetes", "diabetes_stage"], axis=1)
y = df_final['diagnosed_diabetes']

# Split data (80/20) - 100,000 rows
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

def log_metrics(y_true, y_pred, duration):
    """Log performance metrics to MLflow"""
    mlflow.log_metric("recall", recall_score(y_true, y_pred))
    mlflow.log_metric("accuracy", accuracy_score(y_true, y_pred))
    mlflow.log_metric("f1", f1_score(y_true, y_pred))
    mlflow.log_metric("fit_time", duration)

# ---------------------------------------------------------
# RUN 1: BASELINE 
# ---------------------------------------------------------
with mlflow.start_run(run_name="GBC_Baseline_Defaults"):
    # Calling the classifier with no arguments uses absolute defaults
    gb_base = GradientBoostingClassifier(random_state=42)
    
    start_time = time.time()
    gb_base.fit(X_train, y_train)
    duration = time.time() - start_time
    
    y_pred = gb_base.predict(X_test)
    
    # Log default params
    mlflow.log_params(gb_base.get_params())
    mlflow.log_param("optimization", "none_default")
    log_metrics(y_test, y_pred, duration)

# ---------------------------------------------------------
# RUN 2: GRIDSEARCHCV 
# ---------------------------------------------------------
with mlflow.start_run(run_name="GBC_GridSearch"):
    # Grid testing specific values for estimators, rate, depth and subsample
    param_grid = {
        'n_estimators': [50, 100, 150, 200, 300],
        'learning_rate': [0.01, 0.1, 0.3],
        'max_depth': [2, 4, 6, 8, 10],
        'subsample': [0.7, 0.85, 1.0]
    }
    
    grid = GridSearchCV(
        GradientBoostingClassifier(random_state=42),
        param_grid, cv=3, scoring='recall', n_jobs=-1
    )
    
    start_time = time.time()
    grid.fit(X_train, y_train)
    duration = time.time() - start_time
    
    y_pred_grid = grid.best_estimator_.predict(X_test)
    
    mlflow.log_params(grid.best_params_)
    mlflow.log_param("optimization", "GridSearchCV")
    log_metrics(y_test, y_pred_grid, duration)

# ---------------------------------------------------------
# RUN 3: OPTUNA 
# ---------------------------------------------------------
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 10),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0)
    }
    
    model = GradientBoostingClassifier(**params)
    # Cross-validation focusing on Recall
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='recall', n_jobs=-1).mean()
    return score

with mlflow.start_run(run_name="GBC_Optuna"):
    study = optuna.create_study(direction="maximize")
    start_time = time.time()
    study.optimize(objective, n_trials=12) 
    duration = time.time() - start_time
    
    # Train final champion model
    best_gbc = GradientBoostingClassifier(**study.best_params, random_state=42)
    best_gbc.fit(X_train, y_train)
    
    mlflow.log_params(study.best_params)
    mlflow.log_param("optimization", "optuna")
    log_metrics(y_test, best_gbc.predict(X_test), duration)

[I 2026-05-07 13:36:52,586] A new study created in memory with name: no-name-ffba6d4b-696b-4578-961e-033e1cb33e8f
[I 2026-05-07 13:38:07,483] Trial 0 finished with value: 0.8690570165843697 and parameters: {'n_estimators': 224, 'learning_rate': 0.011547335723122274, 'max_depth': 7, 'subsample': 0.927484285239454}. Best is trial 0 with value: 0.8690570165843697.
[I 2026-05-07 13:38:56,932] Trial 1 finished with value: 0.869182020490864 and parameters: {'n_estimators': 144, 'learning_rate': 0.04966205371223443, 'max_depth': 6, 'subsample': 0.9256583836332702}. Best is trial 1 with value: 0.869182020490864.
[I 2026-05-07 13:39:17,080] Trial 2 finished with value: 0.8691195191886992 and parameters: {'n_estimators': 70, 'learning_rate': 0.06803621380488373, 'max_depth': 5, 'subsample': 0.9737198317562574}. Best is trial 1 with value: 0.869182020490864.
[I 2026-05-07 13:41:21,806] Trial 3 finished with value: 0.8695362014813427 and parameters: {'n_estimators': 249, 'learning_rate': 0.0482664

## Runs Summary

| Run | n_estimators | learning_rate | max_depth | subsample | Accuracy | F1 | Recall | Fit Time |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| GBC_Baseline_Defaults | 100 | 0.1 | 3 | 1.0 | 0.91985 | 0.9284343051 | 0.86650 | 28.39s |
| GBC_GridSearch | 50 | 0.3 | 10 | 0.7 | 0.90440 | 0.9164700743 | 0.87408 | 4503.13s |
| GBC_Optuna | 130 | 0.1983281732 | 7 | 0.8766413523 | 0.91645 | 0.9258815702 | 0.86975 | 931.36s |

### Additional logged parameters
- `random_state = 42` where set in code
- Hyperparameters optimized: `n_estimators`, `learning_rate`, `max_depth`, `subsample`

## Best Run Justification for Streamlit

The best run for Streamlit is **GBC_Baseline_Defaults**. Across the three executions, it offers the best balance between metrics and computational cost: it delivers the **highest Accuracy**, the **highest F1**, and a sufficiently competitive Recall, without penalizing training time.

The **GBC_GridSearch** achieved the highest Recall, but that came with a drop in Accuracy and F1 and a very high training cost, which does not make sense for a Streamlit application. The **GBC_Optuna** run slightly improved F1 compared with GridSearch, but it still stayed below the baseline in Accuracy and Recall while keeping a training time much higher than the baseline.

Thus, for a stable diabetes prediction in Streamlit, the most balanced choice is **GBC_Baseline_Defaults**:
- **Accuracy**: 0.91985
- **F1**: 0.92843
- **Recall**: 0.86650
- **Fit time**: 28.39s
